In [1]:
import micropip
await micropip.install("pandas")

In [2]:
import pandas as pd

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
movies.shape, ratings.shape

((9742, 3), (100836, 4))

In [5]:
movie_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating")

movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
toy_story_ratings = movie_matrix[1]

toy_story_ratings.head()

userId
1    4.0
2    NaN
3    NaN
4    NaN
5    4.0
Name: 1, dtype: float64

In [7]:
similar_movies = movie_matrix.corrwith(toy_story_ratings)

similar_movies.head()

/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


movieId
1    1.000000
2    0.330978
3    0.487109
4    1.000000
5    0.310971
dtype: float64

In [8]:
corr_toy_story = pd.DataFrame(similar_movies, columns=["Correlation"])

corr_toy_story.dropna(inplace=True)

corr_toy_story.sort_values("Correlation", ascending=False).head(10)

,Correlation
movieId,
1,1.0
48596,1.0
6992,1.0
4617,1.0
4619,1.0
86817,1.0
209,1.0
4632,1.0
85885,1.0


In [9]:
movie_titles = movies.set_index("movieId")

corr_toy_story = corr_toy_story.join(movie_titles)

corr_toy_story.sort_values("Correlation", ascending=False).head(10)

,Correlation,title,genres
movieId,,,
1,1.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
48596,1.0,"Marine, The (2006)",Action|Drama|Thriller
6992,1.0,Guarding Tess (1994),Comedy|Drama
4617,1.0,Let It Ride (1989),Comedy
4619,1.0,Little Monsters (1989),Comedy
86817,1.0,Something Borrowed (2011),Comedy|Drama|Romance
209,1.0,White Man's Burden (1995),Drama
4632,1.0,"Package, The (1989)",Action|Thriller
85885,1.0,Room in Rome (Habitación en Roma) (2010),Drama|Romance


In [10]:
ratings_count = ratings.groupby("movieId").count()["rating"]

corr_toy_story["rating_count"] = ratings_count

corr_toy_story.head()

,Correlation,title,genres,rating_count
movieId,,,,
1,1.000000,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215
2,0.330978,Jumanji (1995),Adventure|Children|Fantasy,110
3,0.487109,Grumpier Old Men (1995),Comedy|Romance,52
4,1.000000,Waiting to Exhale (1995),Comedy|Drama|Romance,7
5,0.310971,Father of the Bride Part II (1995),Comedy,49


In [11]:
corr_toy_story[corr_toy_story["rating_count"] > 100] \
.sort_values("Correlation", ascending=False) \
.head(10)

,Correlation,title,genres,rating_count
movieId,,,,
1,1.000000,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215
8961,0.643301,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,125
6377,0.618701,Finding Nemo (2003),Adventure|Animation|Children|Comedy,141
588,0.611892,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,183
4886,0.490231,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,132
500,0.446261,Mrs. Doubtfire (1993),Comedy|Drama,144
4973,0.438237,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,120
2706,0.420117,American Pie (1999),Comedy|Romance,103
165,0.410939,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,144


In [12]:
def recommend(movie_id):

    movie_ratings = movie_matrix[movie_id]

    similar = movie_matrix.corrwith(movie_ratings)

    corr_df = pd.DataFrame(similar, columns=["Correlation"])

    corr_df.dropna(inplace=True)

    corr_df["rating_count"] = ratings.groupby("movieId").count()["rating"]

    recommendations = corr_df[corr_df["rating_count"] > 100] \
        .sort_values("Correlation", ascending=False) \
        .head(10)

    return recommendations

In [13]:
recommend(1)

/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


,Correlation,rating_count
movieId,,
1,1.000000,215
8961,0.643301,125
6377,0.618701,141
588,0.611892,183
4886,0.490231,132
500,0.446261,144
4973,0.438237,120
2706,0.420117,103
165,0.410939,144


In [14]:
recommendations = recommend(1)

recommendations = recommendations.join(movies.set_index("movieId"))

recommendations

/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


,Correlation,rating_count,title,genres
movieId,,,,
1,1.000000,215,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
8961,0.643301,125,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy
6377,0.618701,141,Finding Nemo (2003),Adventure|Animation|Children|Comedy
588,0.611892,183,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
4886,0.490231,132,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
500,0.446261,144,Mrs. Doubtfire (1993),Comedy|Drama
4973,0.438237,120,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
2706,0.420117,103,American Pie (1999),Comedy|Romance
165,0.410939,144,Die Hard: With a Vengeance (1995),Action|Crime|Thriller
